In [ ]:
from shutil import copytree

import coremltools as ct
import numpy as np
import torch
from transformers import AutoTokenizer

from export import StatefulModelForCausalLM
from generate import generate

In [17]:
batch_size, context_size = 1, 2048

In [ ]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
device = 'cpu'

# Initialize and trace PyTorch model
torch_model: torch.nn.Module = StatefulModelForCausalLM(
    model_id, batch_size=batch_size, max_context_size=context_size
).eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [19]:
example_inputs: tuple[torch.Tensor, ...] = (
    torch.zeros((1, 2), dtype=torch.int32, device=device),
    torch.zeros((1, 1, 2, 5), dtype=torch.float32, device=device),
)
traced_model: torch.jit.ScriptModule = torch.jit.trace(
    torch_model.eval(), example_inputs=example_inputs
)

/Users/Sina.Sajadmanesh/Projects/swift-transformers/Examples/Mistral7B/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and is_causal


In [20]:
# Convert to Core ML
query_size = ct.RangeDim(lower_bound=1, upper_bound=context_size, default=1)
final_step = ct.RangeDim(lower_bound=1, upper_bound=context_size, default=1)
inputs: list[ct.TensorType] = [
    ct.TensorType(shape=(batch_size, query_size), dtype=np.int32, name="inputIds"),
    ct.TensorType(
        shape=(batch_size, 1, query_size, final_step),
        dtype=np.float16,
        name="causalMask",
    ),
]
states: list[ct.StateType] = [
    ct.StateType(
        wrapped_type=ct.TensorType(shape=torch_model.kv_cache_shape, dtype=np.float16),
        name="keyCache",
    ),
    ct.StateType(
        wrapped_type=ct.TensorType(shape=torch_model.kv_cache_shape, dtype=np.float16),
        name="valueCache",
    ),
]
outputs: list[ct.TensorType] = [ct.TensorType(dtype=np.float16, name="logits")]
mlmodel: ct.models.MLModel = ct.convert(
    traced_model,
    inputs=inputs,
    outputs=outputs,
    states=states,
    minimum_deployment_target=ct.target.macOS15,
    compute_units=ct.ComputeUnit.ALL,
    compute_precision=ct.precision.FLOAT16,
)
mlmodel.save(f'models/{model_id}.mlpackage')

Torch var valueCache is added again.
Torch var keyCache is added again.
Running MIL default pipeline:  67%|██████▋   | 64/95 [00:30<00:36,  1.17s/ passes]/Users/Sina.Sajadmanesh/Projects/swift-transformers/Examples/Mistral7B/.venv/lib/python3.12/site-packages/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur_range.high, tmp_range.high)
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 63.47 passes/s]


In [ ]:
compiled_model_path = mlmodel.get_compiled_model_path()
copytree(compiled_model_path, f'models/{model_id}.mlmodelc', dirs_exist_ok=True)
mlmodel = ct.models.CompiledMLModel(f'models/{model_id}.mlmodelc', optimization_hints={ 'specializationStrategy': ct.SpecializationStrategy.FastPrediction })

In [9]:
generate(
    mlmodel,
    prompt="Tell me a true short story about France.",
    tokenizer=tokenizer,
    max_new_tokens=128,
)

Time to first token: 338.98 seconds
decode throughput: 22.71 tok/s


'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 28 May 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell me a true short story about France.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nIt was a chilly winter morning in the French countryside, and the snow-covered fields of Provence stretched out as far as the eye could see. The village of Saint-Pierre, nestled in the heart of the Luberon region, was bustling with activity as the locals prepared for the annual Fête de la Musique, a celebration of music and community.\n\nIn the town square, a young musician named Sophie had set up her guitar and began to play a lively tune on the street corner. The music was infectious, and soon, people of all ages gathered around her, tapping their feet and swaying to the rhythm.\n\nAs the sun rose'